# SAM + CLIP

In [1]:
!pip install torch opencv-python Pillow
!pip install git+https://github.com/openai/CLIP.git
!pip install git+https://github.com/facebookresearch/sam2.git


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-ol9_sm4c
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-ol9_sm4c
  Resolved https://github.com/openai/CLIP.git to commit dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1
  Preparing metadata (setup.py) ... done
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369490 sha256=b1215d4d8452942d278ede22afa550803a1d5c55169a7325b07d855eaa01026c
  Stored in directory: /tmp/pip-ephem-wheel-cache-dpoastpa/wheels/35/3e/df/3d24cbfb3b6a06f17a2bfd7d1138900d4365d9028aa8f6e92f
Successfully built clip

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip
  Cloning https://github.com/facebookresearch/sam2.git to /tmp/pip-req-build-wi3_1glm
  Running command git clone --filter=blob:none --quiet https

Mounted at /content/drive


In [2]:
!mkdir -p ../checkpoints/
!wget -P ../checkpoints/ https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt

--2025-03-17 14:04:20--  https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 3.168.167.115, 3.168.167.7, 3.168.167.13, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|3.168.167.115|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 898083611 (856M) [application/vnd.snesdev-page-table]
Saving to: ‘../checkpoints/sam2.1_hiera_large.pt’

sam2.1_hiera_large. 100%[===================>] 856.48M  57.0MB/s    in 15s     

2025-03-17 14:04:36 (56.2 MB/s) - ‘../checkpoints/sam2.1_hiera_large.pt’ saved [898083611/898083611]



In [3]:
import cv2
import torch
import numpy as np
from PIL import Image, ImageDraw

import clip
from sam2.build_sam import build_sam2
from sam2.automatic_mask_generator import SAM2AutomaticMaskGenerator

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [6]:
def convert_box_xywh_to_xyxy(box):
    x1 = box[0]
    y1 = box[1]
    x2 = box[0] + box[2]
    y2 = box[1] + box[3]
    return [x1, y1, x2, y2]

In [7]:
def segment_image(image_array, segmentation_mask):

    segmented_image_array = np.zeros_like(image_array)
    segmented_image_array[segmentation_mask] = image_array[segmentation_mask]
    segmented_image = Image.fromarray(segmented_image_array)

    image_size = (image_array.shape[1], image_array.shape[0])
    black_image = Image.new("RGB", image_size, (0, 0, 0))

    transparency_mask = np.zeros_like(segmentation_mask, dtype=np.uint8)
    transparency_mask[segmentation_mask] = 255
    transparency_mask_image = Image.fromarray(transparency_mask, mode='L')

    black_image.paste(segmented_image, mask=transparency_mask_image)

    return black_image

In [8]:
# sam_checkpoint = "/content/drive/MyDrive/URP/checkpoints/sam2.1_hiera_base_plus.pt"
# model_cfg = "/content/drive/MyDrive/URP/sam2.1_hiera_b+.yaml"
sam2_checkpoint = "../checkpoints/sam2.1_hiera_large.pt"
model_cfg = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2 = build_sam2(model_cfg, sam2_checkpoint, device=device, apply_postprocessing=False)
mask_generator = SAM2AutomaticMaskGenerator(
    model=sam2,
    # points_per_side=64,
    # points_per_batch=128,
    # pred_iou_thresh=0.7,
    # stability_score_thresh=0.92,
    # stability_score_offset=0.7,
    # crop_n_layers=1,
    # box_nms_thresh=0.7,
    # crop_n_points_downscale_factor=2,
    # min_mask_region_area=25.0,
    # use_m2m=True,
    )

model, preprocess = clip.load("ViT-B/32", device=device)

100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 59.7MiB/s]


In [9]:
image_path = "./images/truck.jpg"

image = cv2.imread(image_path)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

masks = mask_generator.generate(image)

cropped_boxes = []

for mask in masks:
    cropped_boxes.append(segment_image(image, mask["segmentation"]).crop(convert_box_xywh_to_xyxy(mask["bbox"])))

In [10]:
@torch.no_grad()
def retriev(elements: list[Image.Image], search_text: str) -> int:

    preprocessed_images = [preprocess(image).to(device) for image in elements]

    tokenized_text = clip.tokenize([search_text]).to(device)

    stacked_images = torch.stack(preprocessed_images)

    image_features = model.encode_image(stacked_images)
    text_features = model.encode_text(tokenized_text)

    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)

    probs = 100. * image_features @ text_features.T
    probs.sort(dim=1, descending=True)

    return probs[:, 0].softmax(dim=0)

In [10]:
# @torch.no_grad()
# def retriev(elements: list[Image.Image], search_text: str) -> int:

#     preprocessed_images = [preprocess(image).to(device) for image in elements]

#     tokenized_text = clip.tokenize([search_text]).to(device)

#     image_features = model.encode_image(preprocessed_images)
#     text_features = model.encode_text(tokenized_text)

#     logits_per_image, logits_per_text = model(preprocessed_images, search_text)
#     probs = logits_per_image.softmax(dim=-1).cpu().numpy()

#     # image_features /= image_features.norm(dim=-1, keepdim=True)
#     # text_features /= text_features.norm(dim=-1, keepdim=True)

#     # probs = 100. * image_features @ text_features.T
#     # probs.sort(dim=1, descending=True)

#     return probs[:,0]
#     # probs[:, 0].softmax(dim=0)

In [11]:
def get_indices_of_values_above_threshold(values, threshold):
    return [i for i, v in enumerate(values) if v > threshold]

In [12]:
text_prompt = "truck"
# text_prompt = "blue car"

In [ ]:
scores = retriev(cropped_boxes, text_prompt)
indices = get_indices_of_values_above_threshold(scores, 0.05)

segmentation_masks = []

for seg_idx in indices:
    segmentation_mask_image = Image.fromarray(masks[seg_idx]["segmentation"].astype('uint8') * 255)
    segmentation_masks.append(segmentation_mask_image)

original_image = Image.open(image_path)
overlay_image = Image.new('RGBA', original_image.size, (0, 0, 0, 0))
overlay_color = (255, 0, 0, 200)

draw = ImageDraw.Draw(overlay_image)
for segmentation_mask_image in segmentation_masks:
    draw.bitmap((0, 0), segmentation_mask_image, fill=overlay_color)

result_image = Image.alpha_composite(original_image.convert('RGBA'), overlay_image)
result_image.save("./result/truck.png")
result_image

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/URP/result/truck.png'